# Beyond Black-Box Models: An Interpretable Two-Stage Framework for ECG Rhythm Monitoring

**Authors:** Atin Mukherjee, Anand Kumar Mukhopadhyay, Nilanjan Biswas, and Suman Samui

**Affiliation:** National Institute of Technology Durgapur

---

## Framework Overview

| Component | Detail |
|-----------|--------|
| **Stage-1** | Shallow RF (depth=8) → 3-class coarse: NSR / Atrial / Ventricular |
| **Uncertainty Gate** | p_max, margin, entropy → selective Stage-2 invocation |
| **Stage-2** | Deep RF/DART → 5-class fine: N / S / V / F / Q |
| **Features** | 24 Time-Domain Morphological Features (TDMFs) |
| **Selection** | Triple-ranking (ANOVA + MI + RF) + correlation filter → 14 features |
| **Augmentation** | 14 TDMFs + 3 Stage-1 OOF probabilities = 17 features |
| **Optimization** | Multi-Objective Bayesian Optimization (MOBO) via Optuna |
| **Explainability** | TreeSHAP + XAI Packet + prototype distances |
| **Edge Target** | ESP32 (240 MHz, 520 KB SRAM, 4 MB Flash) |
| **Dataset** | MIT-BIH Arrhythmia (Kaggle 5-class format) |

In [ ]:
# ============================================================
# Cell 1: Imports & Configuration
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    matthews_corrcoef, confusion_matrix, classification_report,
    cohen_kappa_score, roc_auc_score
)
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.feature_selection import f_classif, mutual_info_classif
from scipy.stats import skew, kurtosis, entropy as sp_entropy

import xgboost as xgb
import lightgbm as lgb

SEED = 42
rng = np.random.RandomState(SEED)

# 5-class mapping (MIT-BIH Kaggle format)
CLASS_NAMES_5 = {0: 'N', 1: 'S', 2: 'V', 3: 'F', 4: 'Q'}
CLASS_NAMES_3 = {0: 'NSR', 1: 'Atrial', 2: 'Ventricular'}

# 5-class → 3-class coarse mapping
COARSE_MAP = {0: 0, 1: 1, 2: 2, 3: 2, 4: 0}  # N,Q→NSR; S→Atrial; V,F→Ventricular

print('Imports complete. SEED =', SEED)

In [ ]:
# ============================================================
# Cell 2: Load MIT-BIH Data (Kaggle 5-class format)
# ============================================================
# Download from: https://www.kaggle.com/datasets/shayanfazeli/heartbeat
# Place mitbih_train.csv and mitbih_test.csv in ./data/

import os

DATA_DIR = './data'
train_path = os.path.join(DATA_DIR, 'mitbih_train.csv')
test_path = os.path.join(DATA_DIR, 'mitbih_test.csv')

df_train = pd.read_csv(train_path, header=None)
df_test = pd.read_csv(test_path, header=None)

X_train = df_train.iloc[:, :-1].values  # 187 samples per beat
y_train = df_train.iloc[:, -1].values.astype(int)
X_test = df_test.iloc[:, :-1].values
y_test = df_test.iloc[:, -1].values.astype(int)

print(f'Training set: {X_train.shape[0]:,} beats × {X_train.shape[1]} samples')
print(f'Test set:     {X_test.shape[0]:,} beats × {X_test.shape[1]} samples')
print(f'\nClass distribution (train):')
for c in range(5):
    n = np.sum(y_train == c)
    print(f'  {CLASS_NAMES_5[c]}: {n:>6,} ({n/len(y_train)*100:.1f}%)')

In [ ]:
# ============================================================
# Cell 3: Visualize Sample Beats per Class
# ============================================================
fig, axes = plt.subplots(1, 5, figsize=(20, 3), sharey=True)
for c in range(5):
    idx = np.where(y_train == c)[0][:3]
    for i in idx:
        axes[c].plot(X_train[i], alpha=0.6, linewidth=0.8)
    axes[c].set_title(f'Class {c}: {CLASS_NAMES_5[c]}')
    axes[c].set_xlabel('Sample')
    if c == 0:
        axes[c].set_ylabel('Amplitude')
plt.suptitle('MIT-BIH Arrhythmia — Sample Beats per Class', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 4: 24 Time-Domain Morphological Features (TDMFs)
# ============================================================
#
# Feature Groups:
#   1. R-peak features (amplitude, position, prominence)
#   2. Q/S valley features (amplitude, position relative to R)
#   3. QRS complex features (width, area, slope)
#   4. Beat morphology (pre/post R ratio, symmetry)
#   5. Statistical features (mean, std, skew, kurtosis, energy)
#   6. Temporal dynamics (RR interval proxies, max diff)
#

def extract_tdmf_onebeat(beat):
    """Extract 24 TDMFs from a single 187-sample beat."""
    features = {}
    n = len(beat)
    mid = n // 2

    # ---- R-peak features ----
    r_idx = np.argmax(beat)
    r_amp = beat[r_idx]
    features['r_peak_amplitude'] = r_amp
    features['r_peak_position'] = r_idx / n  # normalized position

    # R-peak prominence (R - mean of neighbors)
    left_val = beat[max(0, r_idx - 5):r_idx].mean() if r_idx > 0 else 0
    right_val = beat[r_idx+1:min(n, r_idx + 6)].mean() if r_idx < n-1 else 0
    features['r_peak_prominence'] = r_amp - (left_val + right_val) / 2

    # ---- Q and S valleys ----
    # Q-wave: minimum before R-peak (in first half of beat)
    q_search = beat[max(0, r_idx - 30):r_idx] if r_idx > 0 else beat[:1]
    q_idx_local = np.argmin(q_search)
    q_idx = max(0, r_idx - 30) + q_idx_local
    q_amp = beat[q_idx]
    features['q_amplitude'] = q_amp
    features['qr_interval'] = (r_idx - q_idx) / n

    # S-wave: minimum after R-peak
    s_search = beat[r_idx:min(n, r_idx + 30)] if r_idx < n else beat[-1:]
    s_idx_local = np.argmin(s_search)
    s_idx = r_idx + s_idx_local
    s_amp = beat[s_idx]
    features['s_amplitude'] = s_amp
    features['rs_interval'] = (s_idx - r_idx) / n

    # ---- QRS complex features ----
    features['qrs_width'] = (s_idx - q_idx) / n
    qrs_segment = beat[q_idx:s_idx+1]
    features['qrs_area'] = np.trapz(np.abs(qrs_segment)) / n if len(qrs_segment) > 1 else 0

    # QRS slopes
    if r_idx > q_idx:
        features['qrs_upslope'] = (r_amp - q_amp) / max(1, r_idx - q_idx)
    else:
        features['qrs_upslope'] = 0
    if s_idx > r_idx:
        features['qrs_downslope'] = (r_amp - s_amp) / max(1, s_idx - r_idx)
    else:
        features['qrs_downslope'] = 0

    # ---- Beat morphology ----
    pre_r_energy = np.sum(beat[:r_idx]**2) if r_idx > 0 else 0
    post_r_energy = np.sum(beat[r_idx:]**2)
    total_energy = pre_r_energy + post_r_energy
    features['pre_r_ratio'] = pre_r_energy / (total_energy + 1e-10)
    features['beat_symmetry'] = abs(0.5 - features['pre_r_ratio'])

    # ---- Statistical features (full beat) ----
    features['beat_mean'] = np.mean(beat)
    features['beat_std'] = np.std(beat)
    features['beat_skewness'] = float(skew(beat))
    features['beat_kurtosis'] = float(kurtosis(beat))
    features['beat_energy'] = np.sum(beat**2) / n

    # ---- Temporal dynamics ----
    diff_beat = np.diff(beat)
    features['max_abs_diff'] = np.max(np.abs(diff_beat))
    features['mean_abs_diff'] = np.mean(np.abs(diff_beat))

    # Zero-crossing rate
    zcr = np.sum(np.diff(np.sign(beat - np.mean(beat))) != 0) / n
    features['zero_crossing_rate'] = zcr

    # RR-interval proxies (from beat boundaries)
    features['rr_pre'] = r_idx / n  # fraction before R
    features['rr_post'] = (n - r_idx) / n  # fraction after R

    return features


def extract_all_tdmf(X):
    """Extract TDMFs for all beats. Returns DataFrame with 24 features."""
    feats = [extract_tdmf_onebeat(beat) for beat in X]
    return pd.DataFrame(feats)


print('Extracting TDMFs...')
import time
t0 = time.time()
X_train_tdmf = extract_all_tdmf(X_train)
X_test_tdmf = extract_all_tdmf(X_test)
t_feat = time.time() - t0
print(f'Done in {t_feat:.1f}s — {X_train_tdmf.shape[1]} features extracted')
print(f'Features: {list(X_train_tdmf.columns)}')

In [ ]:
# ============================================================
# Cell 5: Feature Selection — Triple Ranking + Correlation Filter
# ============================================================
# Method:
#   1. ANOVA F-statistic ranking
#   2. Mutual Information ranking
#   3. RF Gini importance ranking
#   4. Average rank → top features
#   5. Remove highly correlated pairs (ρ > 0.85)

all_features = list(X_train_tdmf.columns)
n_feats = len(all_features)

# ---- ANOVA ----
f_vals, p_vals = f_classif(X_train_tdmf, y_train)
anova_rank = np.argsort(-f_vals)  # higher F → better
anova_ranks = np.empty(n_feats, dtype=int)
anova_ranks[anova_rank] = np.arange(n_feats)

# ---- Mutual Information ----
mi_vals = mutual_info_classif(X_train_tdmf, y_train, random_state=SEED)
mi_rank = np.argsort(-mi_vals)
mi_ranks = np.empty(n_feats, dtype=int)
mi_ranks[mi_rank] = np.arange(n_feats)

# ---- RF Importance ----
temp_rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)
temp_rf.fit(X_train_tdmf, y_train)
rf_imp = temp_rf.feature_importances_
rf_rank = np.argsort(-rf_imp)
rf_ranks = np.empty(n_feats, dtype=int)
rf_ranks[rf_rank] = np.arange(n_feats)

# ---- Average Rank ----
avg_ranks = (anova_ranks + mi_ranks + rf_ranks) / 3.0

# Display ranking table
rank_df = pd.DataFrame({
    'Feature': all_features,
    'ANOVA_Rank': anova_ranks,
    'MI_Rank': mi_ranks,
    'RF_Rank': rf_ranks,
    'Avg_Rank': avg_ranks
}).sort_values('Avg_Rank')

print('Feature Rankings (lower = better):')
print(rank_df.to_string(index=False))

# ---- Take top-18 by average rank ----
TOP_K = 18
top_features = rank_df['Feature'].iloc[:TOP_K].tolist()
print(f'\nTop-{TOP_K} features selected')

In [ ]:
# ============================================================
# Cell 6: Correlation Filter (ρ < 0.85)
# ============================================================
CORR_THRESHOLD = 0.85

corr_matrix = X_train_tdmf[top_features].corr().abs()

# Find highly correlated pairs and drop the one with worse avg rank
to_drop = set()
for i in range(len(top_features)):
    for j in range(i+1, len(top_features)):
        fi, fj = top_features[i], top_features[j]
        if corr_matrix.loc[fi, fj] > CORR_THRESHOLD:
            # Drop the one with higher (worse) average rank
            ri = rank_df[rank_df['Feature'] == fi]['Avg_Rank'].values[0]
            rj = rank_df[rank_df['Feature'] == fj]['Avg_Rank'].values[0]
            drop_feat = fj if rj > ri else fi
            to_drop.add(drop_feat)
            print(f'  Correlated: {fi} ↔ {fj} (ρ={corr_matrix.loc[fi,fj]:.3f}) → drop {drop_feat}')

selected_features = [f for f in top_features if f not in to_drop]
print(f'\nAfter correlation filter: {len(selected_features)} features')
print(f'Selected: {selected_features}')

In [ ]:
# ============================================================
# Cell 7: Correlation Heatmap of Selected Features
# ============================================================
plt.figure(figsize=(10, 8))
sns.heatmap(
    X_train_tdmf[selected_features].corr(),
    annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    square=True, linewidths=0.5
)
plt.title('Correlation Matrix — Selected Features')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 8: Create Stage-1 Coarse Labels (3-class)
# ============================================================
y_train_coarse = np.array([COARSE_MAP[y] for y in y_train])
y_test_coarse = np.array([COARSE_MAP[y] for y in y_test])

print('Coarse label distribution (train):')
for c in range(3):
    n = np.sum(y_train_coarse == c)
    print(f'  {CLASS_NAMES_3[c]}: {n:>6,} ({n/len(y_train_coarse)*100:.1f}%)')

In [ ]:
# ============================================================
# Cell 9: Stage-1 RF Training with Out-of-Fold Probabilities
# ============================================================
# Using OOF probabilities prevents data leakage when feeding
# Stage-1 outputs to Stage-2.

X_s1_train = X_train_tdmf[selected_features].values
X_s1_test = X_test_tdmf[selected_features].values

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Storage for OOF predictions
oof_proba = np.zeros((len(X_s1_train), 3))  # 3 coarse classes
test_proba_accum = np.zeros((len(X_s1_test), 3))

fold_metrics = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_s1_train, y_train_coarse)):
    X_tr, X_va = X_s1_train[tr_idx], X_s1_train[va_idx]
    y_tr, y_va = y_train_coarse[tr_idx], y_train_coarse[va_idx]

    rf_s1 = RandomForestClassifier(
        n_estimators=400, max_depth=8,
        min_samples_leaf=3, max_features='sqrt',
        class_weight='balanced_subsample',
        random_state=SEED, n_jobs=-1
    )
    rf_s1.fit(X_tr, y_tr)

    # OOF predictions (used as Stage-2 input)
    oof_proba[va_idx] = rf_s1.predict_proba(X_va)
    test_proba_accum += rf_s1.predict_proba(X_s1_test)

    # Fold metrics
    pred_va = rf_s1.predict(X_va)
    f1_va = f1_score(y_va, pred_va, average='macro')
    acc_va = accuracy_score(y_va, pred_va)
    fold_metrics.append({'fold': fold+1, 'acc': acc_va, 'f1': f1_va})
    print(f'  Fold {fold+1}: Acc={acc_va:.4f}, Macro-F1={f1_va:.4f}')

# Average test predictions across folds
test_proba_avg = test_proba_accum / N_FOLDS

# Summary
accs = [m['acc'] for m in fold_metrics]
f1s = [m['f1'] for m in fold_metrics]
print(f'\nStage-1 OOF: Acc={np.mean(accs):.3f}±{np.std(accs):.3f}, '
      f'Macro-F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}')

In [ ]:
# ============================================================
# Cell 10: Train Final Stage-1 on Full Training Set
# ============================================================
stage1_final = RandomForestClassifier(
    n_estimators=400, max_depth=8,
    min_samples_leaf=3, max_features='sqrt',
    class_weight='balanced_subsample',
    random_state=SEED, n_jobs=-1
)
stage1_final.fit(X_s1_train, y_train_coarse)

# Stage-1 test predictions
s1_test_pred = stage1_final.predict(X_s1_test)
s1_test_proba = stage1_final.predict_proba(X_s1_test)

s1_test_acc = accuracy_score(y_test_coarse, s1_test_pred)
s1_test_f1 = f1_score(y_test_coarse, s1_test_pred, average='macro')
print(f'Stage-1 Test: Acc={s1_test_acc:.4f}, Macro-F1={s1_test_f1:.4f}')

# Confusion matrix
cm_s1 = confusion_matrix(y_test_coarse, s1_test_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_s1, annot=True, fmt='d', cmap='Blues',
            xticklabels=['NSR','Atrial','Ventricular'],
            yticklabels=['NSR','Atrial','Ventricular'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Stage-1 Confusion Matrix (3-class)')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 11: Stage-2 Feature Augmentation
# ============================================================
# Augment 14 TDMFs with 3 Stage-1 OOF probabilities → 17 features
# Uses out-of-fold predictions to prevent data leakage

# Training: use OOF probabilities
proba_cols = ['s1_prob_NSR', 's1_prob_Atrial', 's1_prob_Ventricular']
df_oof = pd.DataFrame(oof_proba, columns=proba_cols)

X_stage2_train = pd.concat([
    X_train_tdmf[selected_features].reset_index(drop=True),
    df_oof.reset_index(drop=True)
], axis=1)

# Test: use averaged predictions
df_test_proba = pd.DataFrame(test_proba_avg, columns=proba_cols)
X_stage2_test = pd.concat([
    X_test_tdmf[selected_features].reset_index(drop=True),
    df_test_proba.reset_index(drop=True)
], axis=1)

y_stage2 = y_train.copy()

print(f'Stage-2 train features: {X_stage2_train.shape} (14 TDMF + 3 S1 proba = {X_stage2_train.shape[1]})')
print(f'Stage-2 test features:  {X_stage2_test.shape}')
print(f'Feature names: {list(X_stage2_train.columns)}')

In [ ]:
# ============================================================
# Cell 12: Stage-2 Baseline — Random Forest (5-class)
# ============================================================
X2_tr, X2_va, y2_tr, y2_va = train_test_split(
    X_stage2_train.values, y_stage2,
    test_size=0.15, random_state=SEED, stratify=y_stage2
)

stage2_rf = RandomForestClassifier(
    n_estimators=600, max_depth=None,
    min_samples_leaf=1, max_features='sqrt',
    class_weight='balanced_subsample',
    random_state=SEED, n_jobs=-1
)
stage2_rf.fit(X2_tr, y2_tr)

pred_va_rf = stage2_rf.predict(X2_va)
print('Stage-2 RF Validation:')
print(f'  Macro-F1: {f1_score(y2_va, pred_va_rf, average="macro"):.4f}')
print(f'  Accuracy: {accuracy_score(y2_va, pred_va_rf):.4f}')

In [ ]:
# ============================================================
# Cell 13: Stage-2 Comparison — XGBoost DART
# ============================================================
dtrain = xgb.DMatrix(X2_tr, label=y2_tr)
dval = xgb.DMatrix(X2_va, label=y2_va)

xgb_params = {
    'booster': 'dart',
    'objective': 'multi:softprob',
    'num_class': 5,
    'max_depth': 8,
    'learning_rate': 0.1,
    'rate_drop': 0.1,
    'skip_drop': 0.5,
    'sample_type': 'uniform',
    'normalize_type': 'tree',
    'eval_metric': 'mlogloss',
    'seed': SEED,
    'verbosity': 0,
    'nthread': -1
}

xgb_dart = xgb.train(
    xgb_params, dtrain, num_boost_round=500,
    evals=[(dval, 'val')], verbose_eval=0,
    early_stopping_rounds=20
)

pred_va_xgb = np.argmax(xgb_dart.predict(dval), axis=1)
print('XGBoost DART Validation:')
print(f'  Macro-F1: {f1_score(y2_va, pred_va_xgb, average="macro"):.4f}')
print(f'  Accuracy: {accuracy_score(y2_va, pred_va_xgb):.4f}')
print(f'  Best round: {xgb_dart.best_iteration}')

In [ ]:
# ============================================================
# Cell 14: Stage-2 Comparison — LightGBM DART
# ============================================================
lgb_dart = lgb.LGBMClassifier(
    boosting_type='dart', n_estimators=500,
    max_depth=8, learning_rate=0.1,
    drop_rate=0.1,
    class_weight='balanced',
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_dart.fit(
    X2_tr, y2_tr,
    eval_set=[(X2_va, y2_va)],
    callbacks=[lgb.early_stopping(20, verbose=False),
               lgb.log_evaluation(period=0)]
)

pred_va_lgb = lgb_dart.predict(X2_va)
print('LightGBM DART Validation:')
print(f'  Macro-F1: {f1_score(y2_va, pred_va_lgb, average="macro"):.4f}')
print(f'  Accuracy: {accuracy_score(y2_va, pred_va_lgb):.4f}')
print(f'  Best iteration: {lgb_dart.best_iteration_}')

In [ ]:
# ============================================================
# Cell 15: Stage-2 Model Comparison
# ============================================================
models_val = {
    'Random Forest': pred_va_rf,
    'XGBoost DART': pred_va_xgb,
    'LightGBM DART': pred_va_lgb,
}

comparison = []
for name, pred in models_val.items():
    comparison.append({
        'Model': name,
        'Accuracy': accuracy_score(y2_va, pred),
        'Macro-F1': f1_score(y2_va, pred, average='macro'),
        'Weighted-F1': f1_score(y2_va, pred, average='weighted'),
        'MCC': matthews_corrcoef(y2_va, pred),
        'Cohen-κ': cohen_kappa_score(y2_va, pred),
    })

comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))

# Select best model
best_idx = comp_df['Macro-F1'].idxmax()
best_model_name = comp_df.loc[best_idx, 'Model']
print(f'\nBest Stage-2 model (by Macro-F1): {best_model_name}')

In [ ]:
# ============================================================
# Cell 16: Full Test Set Evaluation (Best Model)
# ============================================================
# Evaluate on held-out test set
pred_test = stage2_rf.predict(X_stage2_test.values)

print('Full Test Set Results (Stage-2 RF):')
print('=' * 50)
print(f'Accuracy:    {accuracy_score(y_test, pred_test):.4f}')
print(f'Macro-F1:    {f1_score(y_test, pred_test, average="macro"):.4f}')
print(f'Weighted-F1: {f1_score(y_test, pred_test, average="weighted"):.4f}')
print(f'MCC:         {matthews_corrcoef(y_test, pred_test):.4f}')
print(f'Cohen-κ:     {cohen_kappa_score(y_test, pred_test):.4f}')

# Per-class report
print('\nPer-Class Report:')
print(classification_report(y_test, pred_test,
      target_names=[f'{CLASS_NAMES_5[i]}' for i in range(5)]))

# ROC-AUC (OvR)
try:
    y_test_bin = label_binarize(y_test, classes=[0,1,2,3,4])
    pred_proba_test = stage2_rf.predict_proba(X_stage2_test.values)
    auc_ovr = roc_auc_score(y_test_bin, pred_proba_test, average='macro', multi_class='ovr')
    print(f'ROC-AUC (OvR): {auc_ovr:.4f}')
except:
    print('ROC-AUC: skipped (probability calibration issue)')

In [ ]:
# ============================================================
# Cell 17: Test Confusion Matrix (5-class)
# ============================================================
cm = confusion_matrix(y_test, pred_test)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[CLASS_NAMES_5[i] for i in range(5)],
            yticklabels=[CLASS_NAMES_5[i] for i in range(5)])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Test Confusion Matrix — Two-Stage Framework')
plt.tight_layout()
plt.show()

# Normalized confusion matrix
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Oranges',
            xticklabels=[CLASS_NAMES_5[i] for i in range(5)],
            yticklabels=[CLASS_NAMES_5[i] for i in range(5)])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Normalized Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 18: Multi-Objective Bayesian Optimization (MOBO) — OPTIMIZED
# ============================================================
# Speedups: subsample 15K, 40 trials, cached DMatrix, early stopping

import optuna
from optuna.samplers import TPESampler
import time

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---- Subsample for faster MOBO (stratified) ----
MOBO_TRAIN_SIZE = 15000

if len(X_stage2_train) > MOBO_TRAIN_SIZE:
    X_opt_tr, X_opt_va, y_opt_tr, y_opt_va = train_test_split(
        X_stage2_train.values, y_stage2,
        test_size=0.15, train_size=MOBO_TRAIN_SIZE,
        random_state=SEED, stratify=y_stage2
    )
else:
    X_opt_tr, X_opt_va, y_opt_tr, y_opt_va = train_test_split(
        X_stage2_train.values, y_stage2,
        test_size=0.15, random_state=SEED, stratify=y_stage2
    )

print(f'MOBO train: {X_opt_tr.shape}, val: {X_opt_va.shape}')

# Pre-build XGBoost DMatrix (avoid repeated construction)
dtrain_cached = xgb.DMatrix(X_opt_tr, label=y_opt_tr)
dval_cached   = xgb.DMatrix(X_opt_va, label=y_opt_va)
dval_single   = xgb.DMatrix(X_opt_va[:1])


def mobo_objective(trial):
    """Three-objective: F1 (max), Size (min), Latency (min)."""
    model_type = trial.suggest_categorical('model_type', ['rf', 'xgb_dart', 'lgb_dart'])

    if model_type == 'rf':
        n_est = trial.suggest_int('n_estimators', 50, 500, step=50)
        max_d = trial.suggest_int('max_depth', 4, 16)
        min_leaf = trial.suggest_int('min_samples_leaf', 1, 8)
        max_feat = trial.suggest_categorical('max_features', ['sqrt', 'log2'])

        model = RandomForestClassifier(
            n_estimators=n_est, max_depth=max_d,
            min_samples_leaf=min_leaf, max_features=max_feat,
            random_state=SEED, n_jobs=-1, class_weight='balanced_subsample'
        )
        model.fit(X_opt_tr, y_opt_tr)
        pred = model.predict(X_opt_va)

        t0 = time.perf_counter()
        for _ in range(10):
            model.predict(X_opt_va[:1])
        latency_ms = (time.perf_counter() - t0) / 10 * 1000
        size_proxy = n_est * max_d

    elif model_type == 'xgb_dart':
        n_est = trial.suggest_int('n_estimators', 50, 500, step=50)
        max_d = trial.suggest_int('max_depth', 3, 10)
        lr = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        rate_drop = trial.suggest_float('rate_drop', 0.01, 0.4)
        skip_drop = trial.suggest_float('skip_drop', 0.1, 0.7)

        params = {
            'booster': 'dart', 'objective': 'multi:softprob', 'num_class': 5,
            'max_depth': max_d, 'learning_rate': lr,
            'rate_drop': rate_drop, 'skip_drop': skip_drop,
            'sample_type': 'uniform', 'normalize_type': 'tree',
            'eval_metric': 'mlogloss', 'seed': SEED, 'verbosity': 0, 'nthread': -1
        }

        bst = xgb.train(
            params, dtrain_cached, num_boost_round=n_est,
            evals=[(dval_cached, 'val')], verbose_eval=0,
            early_stopping_rounds=15
        )
        pred = np.argmax(bst.predict(dval_cached), axis=1)

        t0 = time.perf_counter()
        for _ in range(10):
            bst.predict(dval_single)
        latency_ms = (time.perf_counter() - t0) / 10 * 1000
        actual_rounds = bst.best_iteration if bst.best_iteration > 0 else n_est
        size_proxy = actual_rounds * max_d

    else:  # lgb_dart
        n_est = trial.suggest_int('n_estimators', 50, 500, step=50)
        max_d = trial.suggest_int('max_depth', 3, 10)
        lr = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        drop_rate = trial.suggest_float('drop_rate', 0.01, 0.4)

        lgb_m = lgb.LGBMClassifier(
            boosting_type='dart', n_estimators=n_est, max_depth=max_d,
            learning_rate=lr, drop_rate=drop_rate,
            class_weight='balanced', random_state=SEED, n_jobs=-1, verbose=-1
        )
        lgb_m.fit(
            X_opt_tr, y_opt_tr,
            eval_set=[(X_opt_va, y_opt_va)],
            callbacks=[lgb.early_stopping(15, verbose=False), lgb.log_evaluation(period=0)]
        )
        pred = lgb_m.predict(X_opt_va)

        t0 = time.perf_counter()
        for _ in range(10):
            lgb_m.predict(X_opt_va[:1])
        latency_ms = (time.perf_counter() - t0) / 10 * 1000
        actual_rounds = lgb_m.best_iteration_ if lgb_m.best_iteration_ > 0 else n_est
        size_proxy = actual_rounds * max_d

    macro_f1 = f1_score(y_opt_va, pred, average='macro')
    return macro_f1, size_proxy, latency_ms


# ---- Run MOBO ----
N_TRIALS = 40
study = optuna.create_study(
    directions=['maximize', 'minimize', 'minimize'],
    sampler=TPESampler(seed=SEED, multivariate=True),
    study_name='ECG_MOBO'
)

print(f'Running MOBO ({N_TRIALS} trials) on {X_opt_tr.shape[0]:,} training samples...')
t_start = time.time()
study.optimize(mobo_objective, n_trials=N_TRIALS, show_progress_bar=True)
t_mobo = time.time() - t_start

print(f'\nCompleted {len(study.trials)} trials in {t_mobo:.1f}s ({t_mobo/N_TRIALS:.1f}s/trial)')
print(f'Pareto-optimal trials: {len(study.best_trials)}')

In [ ]:
# ============================================================
# Cell 19: Pareto Front Analysis + Top-5 Models
# ============================================================

# Extract Pareto front
pareto_trials = study.best_trials
print(f'Pareto front: {len(pareto_trials)} trials\n')

# All trial results for ranking
trial_results = []
for t in study.trials:
    if t.state == optuna.trial.TrialState.COMPLETE:
        trial_results.append({
            'trial': t.number,
            'model_type': t.params.get('model_type', '?'),
            'f1': t.values[0],
            'size': t.values[1],
            'latency': t.values[2],
            'params': t.params,
            'is_pareto': t in pareto_trials
        })

results_df = pd.DataFrame(trial_results)

# ---- DART Score ----
# Composite: 0.6×norm_F1 − 0.2×norm_Size − 0.2×norm_Latency
f1_min, f1_max = results_df['f1'].min(), results_df['f1'].max()
sz_min, sz_max = results_df['size'].min(), results_df['size'].max()
lt_min, lt_max = results_df['latency'].min(), results_df['latency'].max()

results_df['norm_f1'] = (results_df['f1'] - f1_min) / (f1_max - f1_min + 1e-10)
results_df['norm_size'] = (results_df['size'] - sz_min) / (sz_max - sz_min + 1e-10)
results_df['norm_latency'] = (results_df['latency'] - lt_min) / (lt_max - lt_min + 1e-10)
results_df['dart_score'] = 0.6 * results_df['norm_f1'] - 0.2 * results_df['norm_size'] - 0.2 * results_df['norm_latency']

# ---- Top-5 by DART Score ----
top5 = results_df.nlargest(5, 'dart_score')
print('Top-5 Models by DART Score:')
print('=' * 80)
for i, (_, row) in enumerate(top5.iterrows()):
    print(f'\nRank {i+1}: Trial #{row["trial"]} — {row["model_type"]}')
    print(f'  Macro-F1: {row["f1"]:.4f}  |  Size: {row["size"]:.0f}  |  Latency: {row["latency"]:.3f} ms')
    print(f'  DART Score: {row["dart_score"]:.4f}  |  Pareto: {row["is_pareto"]}')
    print(f'  Hyperparameters: {row["params"]}')

In [ ]:
# ============================================================
# Cell 20: Pareto Front Visualization
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = {'rf': '#2196F3', 'xgb_dart': '#FF5722', 'lgb_dart': '#4CAF50'}

for ax, (x_col, y_col, xlabel, ylabel) in zip(axes, [
    ('f1', 'size', 'Macro-F1 ↑', 'Model Size ↓'),
    ('f1', 'latency', 'Macro-F1 ↑', 'Latency (ms) ↓'),
    ('size', 'latency', 'Model Size ↓', 'Latency (ms) ↓'),
]):
    for mt in ['rf', 'xgb_dart', 'lgb_dart']:
        subset = results_df[results_df['model_type'] == mt]
        pareto_sub = subset[subset['is_pareto']]
        non_pareto = subset[~subset['is_pareto']]
        ax.scatter(non_pareto[x_col], non_pareto[y_col], c=colors[mt], alpha=0.3, s=30, label=f'{mt}')
        ax.scatter(pareto_sub[x_col], pareto_sub[y_col], c=colors[mt], alpha=1.0, s=80, marker='*', edgecolors='k', label=f'{mt} (Pareto)')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)

plt.suptitle('MOBO Pareto Front — Three-Objective Optimization', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 21: Uncertainty Gating Mechanism
# ============================================================
# Criteria for invoking Stage-2:
#   1. p_max < threshold (low confidence)
#   2. margin < threshold (close decision)
#   3. entropy > threshold (high uncertainty)

def compute_uncertainty_metrics(proba):
    """Compute p_max, margin, and entropy from probability vector."""
    sorted_p = np.sort(proba, axis=1)[:, ::-1]
    p_max = sorted_p[:, 0]
    margin = sorted_p[:, 0] - sorted_p[:, 1]
    ent = sp_entropy(proba.T + 1e-10)  # add epsilon for numerical stability
    return p_max, margin, ent


# Compute on test set
p_max, margin, entropy_vals = compute_uncertainty_metrics(s1_test_proba)

# Thresholds (tunable)
PMAX_THRESH = 0.85
MARGIN_THRESH = 0.3
ENTROPY_THRESH = 0.5

# Gate: invoke Stage-2 when ANY uncertainty criterion is met
invoke_mask = (p_max < PMAX_THRESH) | (margin < MARGIN_THRESH) | (entropy_vals > ENTROPY_THRESH)

invoke_rate = invoke_mask.mean()
skip_rate = 1 - invoke_rate
print(f'Uncertainty Gating Results:')
print(f'  p_max < {PMAX_THRESH}:   {(p_max < PMAX_THRESH).mean()*100:.1f}%')
print(f'  margin < {MARGIN_THRESH}: {(margin < MARGIN_THRESH).mean()*100:.1f}%')
print(f'  entropy > {ENTROPY_THRESH}: {(entropy_vals > ENTROPY_THRESH).mean()*100:.1f}%')
print(f'  Stage-2 invoked: {invoke_rate*100:.1f}%')
print(f'  Stage-2 skipped: {skip_rate*100:.1f}%')

In [ ]:
# ============================================================
# Cell 22: Gated Pipeline — Full Evaluation
# ============================================================
# For beats where Stage-2 is skipped, use Stage-1 coarse → fine mapping
# Mapping: NSR(0)→N(0), Atrial(1)→S(1), Ventricular(2)→V(2)
COARSE_TO_FINE = {0: 0, 1: 1, 2: 2}

gated_pred = np.zeros(len(y_test), dtype=int)

# Where Stage-2 is invoked: use Stage-2 predictions
gated_pred[invoke_mask] = pred_test[invoke_mask]

# Where Stage-2 is skipped: use Stage-1 coarse prediction mapped to fine
skip_mask = ~invoke_mask
gated_pred[skip_mask] = np.array([COARSE_TO_FINE[c] for c in s1_test_pred[skip_mask]])

# Evaluate
gated_f1 = f1_score(y_test, gated_pred, average='macro')
gated_acc = accuracy_score(y_test, gated_pred)
full_f1 = f1_score(y_test, pred_test, average='macro')

print(f'Gated Pipeline Results:')
print(f'  Full pipeline Macro-F1:  {full_f1:.4f}')
print(f'  Gated pipeline Macro-F1: {gated_f1:.4f}')
print(f'  F1 delta:                {gated_f1 - full_f1:+.4f}')
print(f'  Stage-2 savings:         {skip_rate*100:.1f}% beats skip Stage-2')

In [ ]:
# ============================================================
# Cell 23: Gating Threshold Sensitivity Analysis
# ============================================================
pmax_thresholds = np.arange(0.5, 1.0, 0.05)
sensitivity_results = []

for pmt in pmax_thresholds:
    mask = (p_max < pmt) | (margin < MARGIN_THRESH) | (entropy_vals > ENTROPY_THRESH)
    g_pred = np.zeros(len(y_test), dtype=int)
    g_pred[mask] = pred_test[mask]
    g_pred[~mask] = np.array([COARSE_TO_FINE[c] for c in s1_test_pred[~mask]])

    sensitivity_results.append({
        'pmax_thresh': pmt,
        'invoke_rate': mask.mean(),
        'macro_f1': f1_score(y_test, g_pred, average='macro'),
        'accuracy': accuracy_score(y_test, g_pred),
    })

sens_df = pd.DataFrame(sensitivity_results)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(sens_df['pmax_thresh'], sens_df['macro_f1'], 'b-o', label='Macro-F1')
ax1.set_xlabel('p_max Threshold')
ax1.set_ylabel('Macro-F1', color='b')

ax2 = ax1.twinx()
ax2.plot(sens_df['pmax_thresh'], sens_df['invoke_rate']*100, 'r--s', label='Stage-2 Invoke %')
ax2.set_ylabel('Stage-2 Invoke Rate (%)', color='r')

plt.title('Gating Sensitivity: p_max Threshold vs. F1 and Invoke Rate')
fig.legend(loc='lower right', bbox_to_anchor=(0.88, 0.15))
plt.tight_layout()
plt.show()

print(sens_df.to_string(index=False))

In [ ]:
# ============================================================
# Cell 24: SHAP Analysis — OPTIMIZED
# ============================================================
# Uses compact RF (depth=12, 200 trees) for fast TreeSHAP.
# TreeSHAP complexity: O(T × L × D²) — depth is quadratic killer.

import shap

shap_features = list(X_stage2_train.columns)

# Compact SHAP-friendly RF (much faster than full-depth model)
print('Training compact RF for SHAP analysis (depth=12, 200 trees)...')
shap_rf = RandomForestClassifier(
    n_estimators=200, max_depth=12,
    min_samples_leaf=2, random_state=SEED,
    n_jobs=-1, class_weight='balanced_subsample'
)
shap_rf.fit(X2_tr, y2_tr)

# Verify accuracy
pred_shap_check = shap_rf.predict(X2_va)
print(f'Compact RF Macro-F1: {f1_score(y2_va, pred_shap_check, average="macro"):.4f} '
      f'(full RF: {f1_score(y2_va, pred_va_rf, average="macro"):.4f})')

# Create explainer
shap_model = shap_rf
explainer = shap.TreeExplainer(shap_model)

# 500 samples (sufficient for all SHAP plots)
SHAP_N = 500
shap_sample_idx = rng.choice(len(X2_va), size=min(SHAP_N, len(X2_va)), replace=False)
X_shap_sample = X2_va[shap_sample_idx]

print(f'Computing SHAP values for {SHAP_N} samples × {len(shap_features)} features × 5 classes...')
t0 = time.time()
shap_values = explainer.shap_values(X_shap_sample)
t_shap = time.time() - t0
print(f'Done in {t_shap:.1f}s')

# ---- Handle SHAP format (list vs 3D array) ----
if isinstance(shap_values, list):
    print(f'SHAP format: list of {len(shap_values)} arrays, each {shap_values[0].shape}')
else:
    print(f'SHAP format: single array {shap_values.shape}')
    if shap_values.ndim == 3:
        shap_values = [shap_values[:, :, c] for c in range(shap_values.shape[2])]
        print(f'Converted to list of {len(shap_values)} arrays')

# Convert to DataFrame (prevents shape mismatch errors)
X_shap_df = pd.DataFrame(X_shap_sample, columns=shap_features)
assert X_shap_df.shape == shap_values[0].shape, 'Shape mismatch!'
print('Shapes verified ✓')

In [ ]:
# ============================================================
# Cell 25: SHAP Global Summary Plot
# ============================================================
plt.figure(figsize=(12, 6))
shap.summary_plot(
    shap_values, X_shap_df,
    feature_names=shap_features,
    class_names=[CLASS_NAMES_5[i] for i in range(5)],
    plot_type='bar',
    show=False
)
plt.title('SHAP Feature Importance (Global — All Classes)')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 26: SHAP Per-Class Beeswarm Plots
# ============================================================
for c in range(5):
    plt.figure(figsize=(10, 5))
    shap.summary_plot(
        shap_values[c],
        X_shap_df,  # DataFrame with correct feature names
        feature_names=shap_features,
        show=False,
        max_display=10
    )
    plt.title(f'SHAP — Class {c}: {CLASS_NAMES_5[c]}')
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# Cell 27: SHAP Force Plot + Waterfall — Single Beat (FIXED)
# ============================================================
# Fixes: same model for explainer+predict, DataFrame Series input

explain_idx = 10
x_explain = X_stage2_test.values[explain_idx:explain_idx+1]

# Use SAME model as explainer
sv_single = explainer.shap_values(x_explain)

# Handle format
if isinstance(sv_single, list):
    pass
elif hasattr(sv_single, 'ndim') and sv_single.ndim == 3:
    sv_single = [sv_single[:, :, c] for c in range(sv_single.shape[2])]
elif hasattr(sv_single, 'ndim') and sv_single.ndim == 2:
    sv_single = [sv_single]

pred_class = int(shap_model.predict(x_explain)[0])
print(f'Beat #{explain_idx}: True={y_test[explain_idx]}, Predicted={pred_class} ({CLASS_NAMES_5[pred_class]})')

# Force plot
plt.figure(figsize=(14, 4))
shap.force_plot(
    explainer.expected_value[pred_class],
    sv_single[pred_class][0],
    pd.DataFrame(x_explain, columns=shap_features).iloc[0],  # Series with names
    feature_names=shap_features,
    matplotlib=True,
    show=False
)
plt.title(f'SHAP Force Plot — Beat #{explain_idx} → {CLASS_NAMES_5[pred_class]}')
plt.tight_layout()
plt.show()

# Waterfall plot (cleaner alternative)
try:
    shap_explanation = shap.Explanation(
        values=sv_single[pred_class][0],
        base_values=explainer.expected_value[pred_class],
        data=x_explain[0],
        feature_names=shap_features
    )
    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(shap_explanation, max_display=12, show=False)
    plt.title(f'SHAP Waterfall — Beat #{explain_idx} → {CLASS_NAMES_5[pred_class]}')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Waterfall plot skipped: {e}')

In [ ]:
# ============================================================
# Cell 28: SHAP Dependence Plots (Top Features)
# ============================================================
# Show how top features interact
mean_abs_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
top_feat_idx = np.argsort(-mean_abs_shap)[:4]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, fi in zip(axes.flat, top_feat_idx):
    shap.dependence_plot(
        fi, shap_values[0], X_shap_df,
        feature_names=shap_features, ax=ax, show=False
    )
plt.suptitle('SHAP Dependence Plots (Class N)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 29: XAI Packet — Explainable Classification Output
# ============================================================
import json

def build_xai_packet(beat_idx, X_raw, X_features_test, stage1_model, stage2_model,
                     shap_explainer, shap_feature_names, uncertainty_metrics):
    """Build complete XAI packet for a single beat."""
    # Extract features
    feats = X_features_test.iloc[beat_idx:beat_idx+1]

    # Stage-1
    s1_proba = stage1_model.predict_proba(feats[selected_features].values)
    s1_pred = int(np.argmax(s1_proba))

    # Uncertainty metrics
    p_max_val = float(s1_proba.max())
    sorted_p = np.sort(s1_proba[0])[::-1]
    margin_val = float(sorted_p[0] - sorted_p[1])
    entropy_val = float(sp_entropy(s1_proba[0] + 1e-10))

    # Gating decision
    needs_stage2 = (p_max_val < PMAX_THRESH or
                    margin_val < MARGIN_THRESH or
                    entropy_val > ENTROPY_THRESH)

    # Stage-2 (if invoked)
    x_s2 = X_stage2_test.values[beat_idx:beat_idx+1]
    s2_proba = stage2_model.predict_proba(x_s2)
    s2_pred = int(np.argmax(s2_proba))

    # SHAP for the beat
    sv_beat = shap_explainer.shap_values(x_s2)
    if isinstance(sv_beat, list):
        pass
    elif hasattr(sv_beat, 'ndim') and sv_beat.ndim == 3:
        sv_beat = [sv_beat[:, :, c] for c in range(sv_beat.shape[2])]

    # Top SHAP contributors for predicted class
    final_class = s2_pred if needs_stage2 else COARSE_TO_FINE[s1_pred]
    shap_vals = sv_beat[final_class][0]
    top_shap_idx = np.argsort(-np.abs(shap_vals))[:5]

    packet = {
        'beat_index': beat_idx,
        'classification': {
            'final_class': int(final_class),
            'class_name': CLASS_NAMES_5[final_class],
            'stage1_coarse': CLASS_NAMES_3[s1_pred],
            'stage2_invoked': bool(needs_stage2),
        },
        'confidence': {
            'stage1_probabilities': {CLASS_NAMES_3[i]: round(float(s1_proba[0][i]), 4) for i in range(3)},
            'stage2_probabilities': {CLASS_NAMES_5[i]: round(float(s2_proba[0][i]), 4) for i in range(5)} if needs_stage2 else None,
        },
        'gating': {
            'p_max': round(p_max_val, 4),
            'margin': round(margin_val, 4),
            'entropy': round(entropy_val, 4),
            'thresholds': {'p_max': PMAX_THRESH, 'margin': MARGIN_THRESH, 'entropy': ENTROPY_THRESH},
            'rationale': ('Stage-2 invoked: ' + ', '.join([
                r for r, c in [
                    (f'low p_max ({p_max_val:.3f} < {PMAX_THRESH})', p_max_val < PMAX_THRESH),
                    (f'narrow margin ({margin_val:.3f} < {MARGIN_THRESH})', margin_val < MARGIN_THRESH),
                    (f'high entropy ({entropy_val:.3f} > {ENTROPY_THRESH})', entropy_val > ENTROPY_THRESH),
                ] if c
            ])) if needs_stage2 else 'High-confidence Stage-1 prediction; Stage-2 skipped'
        },
        'shap_attribution': {
            'top_features': [
                {'feature': shap_feature_names[i], 'shap_value': round(float(shap_vals[i]), 4)}
                for i in top_shap_idx
            ]
        },
        'safety': {
            'model_version': '1.0.0',
            'uncertainty_flag': not needs_stage2 and p_max_val < 0.95,
            'note': 'For clinical decision support only. Not a diagnostic device.'
        }
    }
    return packet


# Build XAI packet for sample beat
xai = build_xai_packet(10, X_test, X_test_tdmf, stage1_final, stage2_rf,
                       explainer, shap_features, None)
print(json.dumps(xai, indent=2))

In [ ]:
# ============================================================
# Cell 30: Latency & Memory Profiling (FIXED)
# ============================================================
# Fix: joblib.dump to temp file instead of pickle.dumps in RAM

import time
import os
import tempfile
import joblib

def measure_model_size_kb(model):
    """Measure serialized model size via temp file (no RAM bloat)."""
    with tempfile.NamedTemporaryFile(suffix='.joblib', delete=False) as f:
        tmp_path = f.name
    try:
        joblib.dump(model, tmp_path, compress=0)
        size_kb = os.path.getsize(tmp_path) / 1024
    finally:
        os.remove(tmp_path)
    return size_kb

def measure_latency_ms(func, n_reps=100):
    """Measure per-call latency with warm-up."""
    func()  # warm-up
    t0 = time.perf_counter()
    for _ in range(n_reps):
        func()
    return (time.perf_counter() - t0) / n_reps * 1000

# ---- Latency ----
single_beat = X_test[0:1]
x_s1 = X_test_tdmf[selected_features].values[0:1]
x_s2 = X_stage2_test.values[0:1]

tdmf_latency_ms   = measure_latency_ms(lambda: extract_tdmf_onebeat(single_beat[0]))
stage1_latency_ms  = measure_latency_ms(lambda: stage1_final.predict_proba(x_s1))
stage2_latency_ms  = measure_latency_ms(lambda: stage2_rf.predict(x_s2))

# ---- Model sizes ----
print('Measuring model sizes...')
stage1_size = measure_model_size_kb(stage1_final)
stage2_size = measure_model_size_kb(stage2_rf)

# ---- Report ----
print('=' * 60)
print('PYTHON PROFILING (sklearn overhead included)')
print('=' * 60)
print(f"\n{'Component':<30} {'Latency (ms)':<15} {'Size (KB)'}")
print('-' * 60)
print(f"{'TDMF Extraction':<30} {tdmf_latency_ms:<15.3f} {'N/A'}")
print(f"{'Stage-1 (RF, depth=8)':<30} {stage1_latency_ms:<15.3f} {stage1_size:.1f}")
print(f"{'Stage-2 (RF, full depth)':<30} {stage2_latency_ms:<15.3f} {stage2_size:.1f}")
print('-' * 60)

full_latency = tdmf_latency_ms + stage1_latency_ms + stage2_latency_ms
gated_latency = tdmf_latency_ms + stage1_latency_ms + invoke_mask.mean() * stage2_latency_ms
total_size = stage1_size + stage2_size

print(f"{'Full Pipeline (per beat)':<30} {full_latency:<15.3f} {total_size:.1f}")
print(f"{'Gated Pipeline (avg)':<30} {gated_latency:<15.3f} {total_size:.1f}")
print(f'\nStage-2 invocation rate: {invoke_mask.mean()*100:.1f}%')
print(f'Latency saving (gated): {(1 - gated_latency/full_latency)*100:.1f}%')

In [ ]:
# ============================================================
# Cell 31: ESP32-Realistic Edge Deployment Metrics
# ============================================================
# Computes realistic metrics from tree structure analysis:
#   - Memory: C-struct layout (7 bytes/internal node, 1 byte/leaf)
#   - Latency: estimated at 240 MHz, 8 cycles/comparison

def compute_rf_edge_metrics(model, model_name='RF'):
    """Compute edge-realistic memory and latency from RF tree structure."""
    n_trees = len(model.estimators_)
    n_classes = model.n_classes_

    total_nodes = 0
    total_leaves = 0
    total_internal = 0
    total_depth = 0
    max_depth = 0
    total_comparisons = 0

    for tree_est in model.estimators_:
        tree = tree_est.tree_
        n_nodes = tree.node_count
        leaves = np.sum(tree.children_left == -1)
        internal = n_nodes - leaves

        total_nodes += n_nodes
        total_leaves += leaves
        total_internal += internal

        depth = tree.max_depth
        total_depth += depth
        max_depth = max(max_depth, depth)
        total_comparisons += np.log2(leaves + 1)

    avg_depth = total_depth / n_trees

    # Memory: C struct layout
    BYTES_PER_INTERNAL = 7  # feature_id(1) + threshold(2) + left(2) + right(2)
    BYTES_PER_LEAF = 1      # argmax class only
    mem_bytes = total_internal * BYTES_PER_INTERNAL + total_leaves * BYTES_PER_LEAF

    # Latency: ESP32 @ 240 MHz
    CLOCK_MHZ = 240
    CYCLES_PER_CMP = 8  # conservative (includes cache miss)
    total_cycles = total_comparisons * CYCLES_PER_CMP
    latency_ms = total_cycles / CLOCK_MHZ / 1000

    print(f'\n{"="*60}')
    print(f'ESP32-REALISTIC METRICS: {model_name}')
    print(f'{"="*60}')
    print(f'  Trees: {n_trees}, Total nodes: {total_nodes:,}, Max depth: {max_depth}')
    print(f'  Memory (C-struct): {mem_bytes:,} bytes = {mem_bytes/1024:.1f} KB')
    print(f'  Comparisons/sample: {total_comparisons:.0f}')
    print(f'  Est. latency: {latency_ms:.4f} ms @ {CLOCK_MHZ} MHz')

    ESP32_SRAM_KB = 520
    ESP32_FLASH_KB = 4096
    print(f'  SRAM ({ESP32_SRAM_KB} KB): {"✓ FITS" if mem_bytes/1024 < ESP32_SRAM_KB else "✗ TOO LARGE"}')
    print(f'  Flash ({ESP32_FLASH_KB} KB): {"✓ FITS" if mem_bytes/1024 < ESP32_FLASH_KB else "✗ TOO LARGE"}')

    return {'mem_kb': mem_bytes/1024, 'latency_ms': latency_ms, 'total_nodes': total_nodes,
            'comparisons': total_comparisons}


# Run on both stages
s1_edge = compute_rf_edge_metrics(stage1_final, 'Stage-1 RF (depth=8, T=400)')
s2_edge = compute_rf_edge_metrics(stage2_rf, 'Stage-2 RF (T=600)')

# TDMF cost estimate
tdmf_ops = 1700  # ~1700 arithmetic ops
tdmf_latency_esp32 = tdmf_ops * 4 / 240 / 1000  # 4 cycles/op @ 240 MHz

# Combined summary
total_mem = s1_edge['mem_kb'] + s2_edge['mem_kb']
full_lat = tdmf_latency_esp32 + s1_edge['latency_ms'] + s2_edge['latency_ms']
gated_lat = tdmf_latency_esp32 + s1_edge['latency_ms'] + invoke_mask.mean() * s2_edge['latency_ms']

print(f'\n{"="*60}')
print('COMPLETE ESP32 PIPELINE SUMMARY')
print(f'{"="*60}')
print(f"{'Component':<35} {'Latency (ms)':<15} {'Size (KB)'}")
print('-' * 65)
print(f"{'TDMF Extraction':<35} {tdmf_latency_esp32:<15.4f} {tdmf_ops*4/1024:.2f}")
print(f"{'Stage-1 RF':<35} {s1_edge['latency_ms']:<15.4f} {s1_edge['mem_kb']:.1f}")
print(f"{'Stage-2 RF':<35} {s2_edge['latency_ms']:<15.4f} {s2_edge['mem_kb']:.1f}")
print('-' * 65)
print(f"{'Full Pipeline':<35} {full_lat:<15.4f} {total_mem:.1f}")
print(f"{'Gated Pipeline ({invoke_mask.mean()*100:.1f}% invoke)':<35} {gated_lat:<15.4f} {total_mem:.1f}")
print(f'\nReal-time budget: ~833 ms/beat (72 bpm)')
print(f'Pipeline uses: {full_lat/833*100:.3f}% of budget')

In [ ]:
# ============================================================
# Cell 32: Prototype Distance Analysis
# ============================================================
from scipy.spatial.distance import cdist

# Compute class prototypes (mean feature vector per class)
prototypes = {}
for c in range(5):
    class_mask = y_stage2 == c
    prototypes[c] = X_stage2_train.values[class_mask].mean(axis=0)

proto_matrix = np.array([prototypes[c] for c in range(5)])

# Distance of test predictions to their predicted class prototype
test_dists = []
for i in range(len(pred_test)):
    d = np.linalg.norm(X_stage2_test.values[i] - prototypes[pred_test[i]])
    test_dists.append(d)
test_dists = np.array(test_dists)

# Prototype distance by class
print('Mean Prototype Distance by Predicted Class:')
for c in range(5):
    mask = pred_test == c
    if mask.sum() > 0:
        print(f'  {CLASS_NAMES_5[c]}: {test_dists[mask].mean():.3f} ± {test_dists[mask].std():.3f}')

# Visualize
plt.figure(figsize=(10, 5))
for c in range(5):
    mask = pred_test == c
    if mask.sum() > 0:
        plt.hist(test_dists[mask], bins=30, alpha=0.5, label=CLASS_NAMES_5[c])
plt.xlabel('Distance to Class Prototype')
plt.ylabel('Count')
plt.title('Prototype Distance Distribution by Class')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 33: Paper-Ready Summary Table
# ============================================================
print('\n' + '=' * 70)
print('PAPER-READY RESULTS SUMMARY')
print('=' * 70)

print('\n--- Stage-1 (3-Class Coarse) ---')
print(f'  Accuracy: {s1_test_acc:.4f}')
print(f'  Macro-F1: {s1_test_f1:.4f}')

print('\n--- Stage-2 Model Comparison (Validation) ---')
print(comp_df.to_string(index=False))

print('\n--- Full Pipeline (Test Set) ---')
print(f'  Accuracy:    {accuracy_score(y_test, pred_test):.4f}')
print(f'  Macro-F1:    {f1_score(y_test, pred_test, average="macro"):.4f}')
print(f'  Weighted-F1: {f1_score(y_test, pred_test, average="weighted"):.4f}')
print(f'  MCC:         {matthews_corrcoef(y_test, pred_test):.4f}')

print('\n--- Per-Class F1 ---')
per_class_f1 = f1_score(y_test, pred_test, average=None)
for c in range(5):
    print(f'  {CLASS_NAMES_5[c]}: {per_class_f1[c]:.4f}')

print('\n--- Uncertainty Gating ---')
print(f'  Stage-2 invoke rate: {invoke_rate*100:.1f}%')
print(f'  Gated Macro-F1:      {gated_f1:.4f}')
print(f'  F1 delta:            {gated_f1 - full_f1:+.4f}')

print('\n--- MOBO ---')
print(f'  Total trials:  {len(study.trials)}')
print(f'  Pareto front:  {len(study.best_trials)} solutions')
if len(top5) > 0:
    best_row = top5.iloc[0]
    print(f'  Best DART Score: {best_row["dart_score"]:.4f} ({best_row["model_type"]})')

print('\n--- Edge Deployment (ESP32-projected) ---')
print(f'  Total model size: {total_mem:.1f} KB')
print(f'  Full latency:     {full_lat:.4f} ms')
print(f'  Gated latency:    {gated_lat:.4f} ms')
print(f'  Budget usage:     {full_lat/833*100:.3f}%')

In [ ]:
# ============================================================
# Cell 34: Save Trained Models
# ============================================================
import joblib
import os

MODEL_DIR = './models'
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(stage1_final, os.path.join(MODEL_DIR, 'stage1_rf.joblib'))
joblib.dump(stage2_rf, os.path.join(MODEL_DIR, 'stage2_rf.joblib'))
xgb_dart.save_model(os.path.join(MODEL_DIR, 'stage2_xgb_dart.json'))
joblib.dump(lgb_dart, os.path.join(MODEL_DIR, 'stage2_lgb_dart.joblib'))

# Save feature names and config
config = {
    'selected_features': selected_features,
    'all_tdmf_features': list(X_train_tdmf.columns),
    'coarse_map': COARSE_MAP,
    'class_names_5': CLASS_NAMES_5,
    'class_names_3': CLASS_NAMES_3,
    'gating_thresholds': {
        'pmax': PMAX_THRESH,
        'margin': MARGIN_THRESH,
        'entropy': ENTROPY_THRESH
    },
    'seed': SEED
}

import json
with open(os.path.join(MODEL_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print(f'Models saved to {MODEL_DIR}/')
for f in os.listdir(MODEL_DIR):
    fpath = os.path.join(MODEL_DIR, f)
    print(f'  {f}: {os.path.getsize(fpath)/1024:.1f} KB')